# Text-to-Speech Generation with OpenVINO GenAI

[OpenVINO™ GenAI](https://github.com/openvinotoolkit/openvino.genai) is a library of the most popular Generative AI model pipelines, optimized execution methods, and samples that run on top of highly performant OpenVINO Runtime.

This library is friendly to PC and laptop execution, and optimized for resource consumption. It requires no external dependencies to run generative models as it already includes all the core functionality (e.g. tokenization via openvino-tokenizers).

Text-to-speech (TTS) technology converts written text into spoken audio. It's a form of speech synthesis that allows users to listen to digital text being read aloud. This technology is used in various applications, including accessibility for those with visual impairments, voice assistants, and language learning tools.

In this notebook we will demonstrate how to use OpenVINO GenAI capabilities for speech synthesis. You can find list of supported by OpenVINO GenAI in [SUPPORTED_MODELS.md](https://github.com/openvinotoolkit/openvino.genai/blob/master/SUPPORTED_MODELS.md#speech-generation-models). In this tutorial we will use [SpeechT5 TTS](https://huggingface.co/microsoft/speecht5_tts) model.
#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert model using Optimum-CLI tool](#Convert-model-using-Optimum-CLI-tool)
- [Prepare inference pipeline](#Prepare-inference-pipeline)
    - [Select inference device](#Select-inference-device)
- [Run inference OpenVINO model with Text2SpeechPipeline](#Run-inference-OpenVINO-model-with-Text2SpeechPipeline)
- [Interactive demo](#Interactive-demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/text-to-speech-genai/text-to-speech-genai.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
import platform

%pip install -U "transformers[sentencepiece]>=4.45" "soundfile" "torch==2.11.*" "torchaudio==2.11.*" "datasets<4.0.0" librosa speechbrain gradio "git+https://github.com/huggingface/optimum-intel.git" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -Uq --pre "openvino>2025.1.0" "openvino-genai" "openvino-tokenizers" --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/pre-release

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0.0"

In [ ]:
from pathlib import Path
import requests

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)


if not Path("gradio_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/text-to-speech-genai/gradio_helper.py",
    )
    open("gradio_helper.py", "w").write(r.text)


if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)


# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("text-to-speech-genai.ipynb")

## Convert model using Optimum-CLI tool
[back to top ⬆️](#Table-of-contents:)

🤗 [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) is the interface between the 🤗 [Transformers](https://huggingface.co/docs/transformers/index) and [Diffusers](https://huggingface.co/docs/diffusers/index) libraries and OpenVINO to accelerate end-to-end pipelines on Intel architectures. It provides ease-to-use cli interface for exporting models to [OpenVINO Intermediate Representation (IR)](https://docs.openvino.ai/2024/documentation/openvino-ir-format.html) format.

The command bellow demonstrates basic command for model export with `optimum-cli`

```bash
optimum-cli export openvino --model <model_id_or_path> --task <task> <out_dir>
```

where `--model` argument is model id from HuggingFace Hub or local directory with model (saved using `.save_pretrained` method), `--task ` is one of [supported task](https://huggingface.co/docs/optimum/exporters/task_manager) that exported model should solve (optional and can be detected if we download model from HuggingFace Hub). 
If model initialization requires to use remote code, `--trust-remote-code` flag additionally should be passed.
You can also apply fp16, 8-bit or 4-bit weight compression on the Linear, Convolutional and Embedding layers when exporting your model with the CLI by setting `--weight-format` to respectively fp16, int8 or int4. This type of optimization allows to reduce the memory footprint and inference latency.

We will use [SpeechT5](https://huggingface.co/microsoft/speecht5_tts) model, this model conversion required to provide additional arguments for loading to include [vocoder](https://huggingface.co/microsoft/speecht5_hifigan) part.




In [ ]:
import json
from cmd_helper import optimum_cli

model_id = "microsoft/speecht5_tts"
model_path = Path(model_id.split("/")[-1])

if not (model_path / "config.json").exists():
    optimum_cli(model_id, "speecht5_tts", additional_args={"model-kwargs": json.dumps({"vocoder": "microsoft/speecht5_hifigan"}, separators=(",", ":"))})

## Prepare inference pipeline
[back to top ⬆️](#Table-of-contents:)

The SpeechT5 framework consists of a shared encoder-decoder network and six modal-specific (speech/text) pre/post-nets. After preprocessing the input speech/text through the pre-nets, the shared encoder-decoder network models the sequence-to-sequence transformation, and then the post-nets generate the output in the speech/text modality based on the output of the decoder.

`opevino_genai.TextToSpeechPipeline` implements inference pipeline for speech generation. It accepts directory with converted model and inference device for initialization.

### Select inference device
[back to top ⬆️](#Table-of-contents:)

In [4]:
from notebook_utils import device_widget

device = device_widget(exclude=["NPU"])
device

Dropdown(description='Device:', index=1, options=('CPU', 'AUTO'), value='AUTO')

In [ ]:
import openvino_genai as ov_genai
import openvino as ov
import IPython.display as ipd


def play(data, rate=None):
    ipd.display(ipd.Audio(data, rate=rate))


def get_audio_data(speech):
    """Extract audio numpy array from ov.Tensor, handling remote tensors (e.g., GPU)."""
    try:
        return speech.data[0]
    except RuntimeError:
        host_tensor = ov.Tensor(speech.element_type, speech.shape)
        speech.copy_to(host_tensor)
        return host_tensor.data[0]


pipe = ov_genai.Text2SpeechPipeline(model_path, device.value)

## Run inference OpenVINO model with Text2SpeechPipeline
[back to top ⬆️](#Table-of-contents:)

for starting generation, you should provide input text to pipeline `generate` method. Inference result is speech tensor contains the waveform of the spoken phrase with sample rate 16000 Hz. You can save its data represented as numpy array using standard audio processing libraries (librosa, torchaudio, soundfile e.t.c.).

In [ ]:
import soundfile as sf

input_text = "It is not in the stars to hold our destiny but in ourselves."

result = pipe.generate(input_text)

speech = result.speeches[0]
audio_data = get_audio_data(speech)
output_file_name = "output_audio.wav"
sf.write(output_file_name, audio_data, samplerate=16000)

play(audio_data, rate=16000)

SpeechT5 pipeline also supports voice cloning. For enabling this feature, you need to provide xvector which represent speaker embeddings. We will use one of prepared voices from [cmu-arctic-xvectors](https://huggingface.co/datasets/Matthijs/cmu-arctic-xvectors) dataset. It is also possible to create own speaker embeddings, for guidance how to do that please check [this sample](https://github.com/openvinotoolkit/openvino.genai/tree/master/samples/python/speech_generation#prepare-speaker-embedding-file) 

In [ ]:
from datasets import load_dataset
import numpy as np

embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
speaker_embedding = ov.Tensor(np.array(embeddings_dataset[7306]["xvector"], dtype=np.float32).reshape(1, -1))

result = pipe.generate(input_text, speaker_embedding)

speech = result.speeches[0]
audio_data = get_audio_data(speech)
output_file_name = "output_audio_with_speaker.wav"
sf.write(output_file_name, audio_data, samplerate=16000)

play(audio_data, rate=16000)

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from gradio_helper import make_demo

demo = make_demo(pipe)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(share=True, debug=True)

# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/